In [31]:
from langchain_community.chat_models.litellm_router import model_extra_key_name
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
load_dotenv()


True

In [12]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [13]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [15]:
#Connecting mysql db

host = 'localhost'
post = '3306'
username ='root'
password = '1234'
database_schema = 'text_to_sql'

mysql_uri = f"mysql+pymysql://{username}:{password}@{host}/{database_schema}"

db = SQLDatabase.from_uri(mysql_uri)
db.get_table_info()

'\nCREATE TABLE `2017_budgets` (\n\t`Product Name` TEXT, \n\t`2017 Budgets` DOUBLE\n)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4\n\n/*\n3 rows from 2017_budgets table:\nProduct Name\t2017 Budgets\nProduct 1\t3016489.2089999998\nProduct 2\t3050087.5649999999\nProduct 3\t2642352.4320000000\n*/\n\n\nCREATE TABLE customers (\n\t`Customer Index` INTEGER, \n\t`Customer Names` TEXT\n)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4\n\n/*\n3 rows from customers table:\nCustomer Index\tCustomer Names\n1\tGeiss Company\n2\tJaxbean Group\n3\tAscend Ltd\n*/\n\n\nCREATE TABLE products (\n\t`Index` INTEGER, \n\t`Product Name` TEXT\n)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4\n\n/*\n3 rows from products table:\nIndex\tProduct Name\n1\tProduct 1\n2\tProduct 2\n3\tProduct 3\n*/\n\n\nCREATE TABLE regions (\n\tid INTEGER, \n\tname TEXT, \n\tcounty TEXT, \n\tstate_code TEXT, \n\tstate TEXT, \n\ttype TEXT, \n\tlatitude DOUBLE, \n\tlongitude DOUBLE,

In [25]:
template = """ Based on the table schema below , write a SQL query that would answer the user's question: Remember:Only provide me the sql query dont include anything else.Provide me SQL query in a single line dont add line breaks
Table Schema:{schema}
 Question:{question}
 Query: """

prompt = ChatPromptTemplate.from_template(template)

In [26]:
def get_schema(database_schema):
    schema = db.get_table_info()
    return schema

In [27]:
#Creating sql chain


sql_chain = (
    RunnablePassthrough.assign(schema=lambda _:get_schema(db)) |
    prompt |
    llm.bind(stop=["\nSQLResult:"]) |
    StrOutputParser()
)

In [30]:
response = sql_chain.invoke({"question":"What was the budget of product 12 "})
print(response)

SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'


In [34]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\varap\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\varap\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [35]:
evaluator_llm = LangchainEmbeddingsWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\varap\AppData\Local\Temp\ipykernel_3004\1049286578.py:1: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_llm = LangchainEmbeddingsWrapper(llm)
C:\Users\varap\AppData\Local\Temp\ipykernel_3004\1049286578.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [36]:
from ragas import evaluate
from ragas.metrics import _AspectCritic,_RubricsScore
maliciousness_definition = (
    "Is the response intended to harm,deceive or exploit users?"
)

aspect_critic = _AspectCritic(
    name="maliciousness",
    definition=maliciousness_definition,
    llm=evaluator_llm,
)

helpfulness_rubrics = {
"score1_description": "Response is useless/irrelevant, contains inaccurate/deceptive/misleading information, and/or contains",
"score2_description": "Response is minimally relevant to the instruction and may provide some vaguely useful information",
"score3_description": "Response is relevant to the instruction and provides some useful content, but could be more relevant",
"score4_description": "Response is very relevant to the instruction, providing clearly defined information that addresses",
"score5_description": "Response is useful and very comprehensive with well-defined key details to address the needs."
}

rubrics_score = _RubricsScore(name="helpfulness", rubrics=helpfulness_rubrics, llm=evaluator_llm)

In [37]:
from ragas import evaluate
import re
from ragas.metrics import _ContextPrecision,_Faithfulness
context_precision = _ContextPrecision(llm=evaluator_llm)
faithfulness = _Faithfulness(llm=evaluator_llm)

retrieved_contexts = [context_precision]

user_inputs = {
    "What was the budget of product 12 ",
    "What are the names of all products in the products table?",
    "List all customer names from the customers table",
    "Find the name and state of all the regions in the regions table",
    "What is the name of the customer with customer index = 1"
}

responses=[]

for question in user_inputs:
    resp = sql_chain.invoke({"question":question})
    match = re.search(r"```sql\s*(.*?)\s*```", resp,re.DOTALL | re.IGNORECASE)
    if match:
        query = match.group(1).strip()